## Tahap 2 — Patching v2: intervensi di TOKEN IDENTITAS (resep finding 07)

v1 (notebook 11) nol karena desain: patch 1-3 head skala natural di token TERAKHIR,
soal acak (langit-langit ~0.004). Bedah kode llm-opinions (finding 07) menunjukkan
output BISA digeser lewat interface yang sama — bedanya di 4 hal. v2 memperbaiki
semuanya:

| aspek | v1 (nol) | v2 (ini) |
|---|---|---|
| Bentuk prompt | identitas = kalimat natural | identitas = **jawaban 1-token** di blok QA demografis (ala llm-opinions) |
| Posisi intervensi | token terakhir (tanpa propagasi) | **posisi token identitas** — perubahan merambat ke semua token setelahnya |
| Kekuatan | 1-3 head, 1x | **semua 32 head L11** (+ varian), skala 1x sebagai instrumen + 5x sebagai pembanding steering |
| Soal | acak | **top-20 per pasangan berdasarkan WD distribusi ASLI terbesar** |

Trik utama: prompt A dan B **identik kecuali 2 token huruf jawaban demografis** —
jadi mencangkok aktivasi di 2 posisi itu = mengganti identitas di sumbernya,
dan `predB` baseline = langit-langit "full swap" yang bisa dibandingkan langsung.

Kondisi:

| kondisi | isi | peran |
|---|---|---|
| `patch_L11_all32` | semua 32 head L11, α=1 | **hipotesis utama** |
| `patch_L11H16` | cuma head bintang, α=1 | apakah 1 head cukup? |
| `patch_L11L18_all` | semua head L11+L18, α=1 | 2 layer |
| `patch_L11_all32_x5` | α=5 (base + 5·(donor−base)) | pembanding gaya steering |
| `patch_randhead` | 1 head acak tetap, α=1 | kontrol negatif |
| `self_patch` | donor = diri sendiri (subset) | sanity (harus ~0) |

Skor tetap: pergeseran WD prediksi ke **distribusi ASLI kelompok donor**
(patching sebagai instrumen kesetiaan, bukan steering).


## Sebelum jalan: setting Kaggle

1. **Accelerator**: GPU T4 x2. **Internet: On**. Attach `opinionqa_intersectional.csv`.
2. Sesi bekas crash -> RESTART SESSION.
3. **Download setelah selesai:** `patching_v2_rows.csv` + `patching_v2_summary.csv`
   dari `/kaggle/working/tahap2_patching_v2/` -> taruh di
   `notebooks/output/12_tahap2_patching_v2_kaggle/`.

Estimasi: ~5.300 forward pass ≈ 1-1.5 jam total.


In [ ]:
!pip install -q -U "transformers>=4.44" accelerate scipy tqdm

In [ ]:
import os, sys, gc, glob, ast
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from scipy.stats import wasserstein_distance, wilcoxon
from tqdm.auto import tqdm

sys.last_traceback = None
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    for d in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(d)
        print(f"GPU {d}: {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")
        if free / total < 0.9:
            print(f"  PERINGATAN: GPU {d} tidak kosong -> RESTART SESSION dulu!")


In [ ]:
MODEL_PATH = "mistralai/Mistral-7B-v0.1"

_candidates = glob.glob("/kaggle/input/**/opinionqa_intersectional.csv", recursive=True)
if _candidates:
    DATA_PATH = _candidates[0]
elif os.path.exists("opinionqa_intersectional.csv"):
    DATA_PATH = "opinionqa_intersectional.csv"
else:
    raise FileNotFoundError("opinionqa_intersectional.csv tidak ketemu.")
print("Data:", DATA_PATH)

RANDOM_SEED = 42
TYPES_RUN = ["AGExPOLPARTY", "RELIGxPOLPARTY", "RACExRELIG"]
N_PAIRS = 12
N_QUESTIONS = 20      # per pasangan, top berdasarkan WD asli terbesar
MAX_OPTIONS = 6       # opsi soal opini (huruf A-F); opsi QA demografis boleh lebih
MIN_SHARED_Q = 20
N_SELF_PATCH = 3

STAR_LAYER, STAR_HEAD = 11, 16
SECOND_LAYER = 18

OUT_DIR = "/kaggle/working/tahap2_patching_v2"
os.makedirs(OUT_DIR, exist_ok=True)


## 1. Data: pasangan + soal ber-perbedaan-asli terbesar

Beda dari v1: soal per pasangan bukan acak, tapi **top-20 WD(realA, realB)** —
soal di mana kedua kelompok BENERAN paling beda pendapat. Ini menaikkan
langit-langit instrumen.


In [ ]:
df = pd.read_csv(DATA_PATH)
for c in ["responses", "ordinal", "options"]:
    df[c] = df[c].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df["group_key"] = df["attribute"] + " :: " + df["group"]
df["n_opt"] = df["ordinal"].apply(len)

qmeta = {}
real_resp = {}
for r in df.itertuples():
    qmeta[r.qkey] = (r.question, r.options[: r.n_opt], r.ordinal)
    real_resp[(r.group_key, r.qkey)] = np.array(r.responses, dtype=np.float64)

def real_wd(A, B, qk):
    _, _, ordinal = qmeta[qk]
    return wasserstein_distance(ordinal, ordinal,
                                u_weights=real_resp[(A, qk)], v_weights=real_resp[(B, qk)])

rng = np.random.default_rng(RANDOM_SEED)
plan = {}
for ty in TYPES_RUN:
    sub = df[(df["attribute"] == ty) & (df["n_opt"] <= MAX_OPTIONS)]
    cells = sorted(sub["group_key"].unique().tolist())
    q_per_cell = sub.groupby("group_key")["qkey"].apply(set).to_dict()
    # nilai unik per komponen -> daftar opsi QA demografis
    v1_opts = sorted({gk.split(" :: ", 1)[1].split(" | ", 1)[0] for gk in cells})
    v2_opts = sorted({gk.split(" :: ", 1)[1].split(" | ", 1)[1] for gk in cells})
    all_pairs = [(a, b) for a in cells for b in cells
                 if a != b and len(q_per_cell[a] & q_per_cell[b]) >= MIN_SHARED_Q]
    pick = rng.choice(len(all_pairs), size=min(N_PAIRS, len(all_pairs)), replace=False)
    pairs = [all_pairs[k] for k in pick]
    pair_questions = {}
    for (a, b) in pairs:
        shared = sorted(q_per_cell[a] & q_per_cell[b])
        wds = sorted(((real_wd(a, b, qk), qk) for qk in shared), reverse=True)
        pair_questions[(a, b)] = [qk for _, qk in wds[:N_QUESTIONS]]
    needed = sorted({(gk, qk) for (a, b), qs in pair_questions.items()
                     for qk in qs for gk in (a, b)})
    plan[ty] = dict(cells=sorted({c for p in pairs for c in p}), pairs=pairs,
                    pair_questions=pair_questions, needed=needed,
                    v1_opts=v1_opts, v2_opts=v2_opts)
    top_wd = np.mean([real_wd(a, b, qk) for (a, b), qs in pair_questions.items() for qk in qs])
    print(f"[{ty}] {len(plan[ty]['cells'])} sel, {len(pairs)} pasangan, baseline unik "
          f"{len(needed)}, mean WD-asli soal terpilih: {top_wd:.3f}")


## 2. Prompt QA-demografis + posisi token identitas

Identitas masuk sebagai **jawaban 1 token** di dua blok QA (satu per komponen),
lalu soal opini. Prompt A vs B identik kecuali 2 token huruf itu -> posisi
intervensi sama persis, dan predB = langit-langit full-swap.


In [ ]:
ATTR_QA = {
    "RACExRELIG":       ("What is this survey respondent's race?",
                         "What is this survey respondent's religion?"),
    "RELIGxPOLPARTY":   ("What is this survey respondent's religion?",
                         "What is this survey respondent's political party affiliation?"),
    "AGExPOLPARTY":     ("What is this survey respondent's age group?",
                         "What is this survey respondent's political party affiliation?"),
}
DEMO_LETTERS = [chr(65 + i) for i in range(26)]
LETTERS = ["A", "B", "C", "D", "E", "F"]

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def demo_block(question, opts, value):
    lines = [f"Question: {question}"]
    for i, o in enumerate(opts):
        lines.append(f"{DEMO_LETTERS[i]}. {o}")
    lines.append(f"Answer: {DEMO_LETTERS[opts.index(value)]}")
    return "\n".join(lines)

def build_prompt(ty, gk, qk):
    v1, v2 = gk.split(" :: ", 1)[1].split(" | ", 1)
    q1, q2 = ATTR_QA[ty]
    p = plan[ty]
    question, options, _ = qmeta[qk]
    blocks = [demo_block(q1, p["v1_opts"], v1), "", demo_block(q2, p["v2_opts"], v2), "",
              f"Question: {question}"]
    for i, opt in enumerate(options):
        blocks.append(f"{LETTERS[i]}) {opt}")
    blocks.append("Answer:")
    return "\n".join(blocks)

def identity_positions(prompt):
    """Posisi token huruf jawaban 2 blok QA demografis (setelah 'Answer:' ke-1 & ke-2)."""
    positions = []
    search_from = 0
    for _ in range(2):
        idx = prompt.find("Answer:", search_from)
        assert idx != -1
        prefix = prompt[: idx + len("Answer:")]
        pos = 1 + len(tokenizer.encode(prefix, add_special_tokens=False))  # +1 BOS
        positions.append(pos)
        search_from = idx + 1
    return positions

# validasi: A vs B beda cuma di 2 token huruf, posisi sama
ty0 = TYPES_RUN[0]
(A0, B0) = plan[ty0]["pairs"][0]
qk0 = plan[ty0]["pair_questions"][(A0, B0)][0]
pA, pB = build_prompt(ty0, A0, qk0), build_prompt(ty0, B0, qk0)
tA = tokenizer(pA, return_tensors="pt")["input_ids"][0]
tB = tokenizer(pB, return_tensors="pt")["input_ids"][0]
posA, posB = identity_positions(pA), identity_positions(pB)
assert len(tA) == len(tB), (len(tA), len(tB))
assert posA == posB, (posA, posB)
diff = (tA != tB).nonzero().flatten().tolist()
print("Prompt contoh:\n" + pA)
print("\nToken beda A vs B di posisi:", diff, "| posisi identitas terdeteksi:", posA)
assert set(diff).issubset(set(posA)), "token beda bukan di posisi identitas!"
for pos in posA:
    print(f"  pos {pos}: A={tokenizer.decode(tA[pos])!r}  B={tokenizer.decode(tB[pos])!r}")


## 3. Load model + mesin patch v2

Patch: `x_patched = x_base + α·(donor − base)` di posisi identitas, per head.
α=1 = swap murni (instrumen); α=5 = amplified (pembanding gaya steering).
Donor diambil dari forward pass prompt B di posisi yang sama.


In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, torch_dtype=torch.float16, device_map="balanced", low_cpu_mem_usage=True
)
model.eval()
NUM_HEADS = model.config.num_attention_heads
HEAD_DIM = model.config.hidden_size // NUM_HEADS

LETTER_IDS = [tokenizer.encode(f" {L}", add_special_tokens=False)[-1] for L in LETTERS]
assert len(set(LETTER_IDS)) == len(LETTER_IDS)

_forbidden = {(STAR_LAYER, STAR_HEAD), (SECOND_LAYER, 14), (11, 19)}
rng_ctrl = np.random.default_rng(RANDOM_SEED + 7)
while True:
    RAND_HEAD = (int(rng_ctrl.integers(0, model.config.num_hidden_layers)),
                 int(rng_ctrl.integers(0, NUM_HEADS)))
    if RAND_HEAD not in _forbidden:
        break
print("Head kontrol acak:", RAND_HEAD)

CAPTURE_LAYERS = sorted({STAR_LAYER, SECOND_LAYER, RAND_HEAD[0]})

_donor_capture = {}   # layer -> {pos: vec4096} (o_proj input)
_capture_positions = []
_active_patch = {}    # layer -> list of (pos, heads, alpha, donor_vec4096)

def _oproj_prehook(layer_idx):
    def fn(module, args):
        x = args[0]
        if _capture_positions:
            # batch-general: simpan SEMUA item di batch, bukan cuma index 0.
            # Konsumen (donors[...]) selalu index eksplisit [b] -> tetap 1D per item,
            # jadi Pass 2 (yang selalu B=1) tidak perlu berubah sama sekali.
            _donor_capture[layer_idx] = {
                pos: x[:, pos, :].detach().float().cpu() for pos in _capture_positions
            }
        patches = _active_patch.get(layer_idx)
        if patches:
            x = x.clone()
            for (pos, heads, alpha, donor) in patches:
                d = donor.to(x.device, x.dtype)
                for h in heads:
                    s = slice(h * HEAD_DIM, (h + 1) * HEAD_DIM)
                    x[0, pos, s] = x[0, pos, s] + alpha * (d[s] - x[0, pos, s])
            return (x,) + tuple(args[1:])
        return None
    return fn

handles = [model.model.layers[L].self_attn.o_proj.register_forward_pre_hook(_oproj_prehook(L))
           for L in CAPTURE_LAYERS]
print("Hook di layer", CAPTURE_LAYERS)

@torch.no_grad()
def forward_pred(prompt, n_opt, capture_positions=None, patch_spec=None):
    """patch_spec: dict layer -> list of (pos, heads, alpha, donor_vec)."""
    global _capture_positions
    _capture_positions = capture_positions or []
    _active_patch.clear()
    if patch_spec:
        _active_patch.update(patch_spec)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    logits = model(**inputs).logits[0, -1, :]
    _active_patch.clear()
    _capture_positions = []
    sel = logits[LETTER_IDS[:n_opt]].float()
    return torch.softmax(sel, dim=0).cpu().numpy()

@torch.no_grad()
def forward_pred_batch(prompts, n_opt, capture_positions):
    """Versi batched forward_pred -- KHUSUS ekstraksi baseline (tanpa patch_spec).
    Prasyarat: semua prompt di `prompts` panjang TOKEN-nya sama (dicek pemanggil).
    """
    global _capture_positions
    _capture_positions = capture_positions
    _active_patch.clear()
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
    logits = model(**inputs).logits[:, -1, :]
    _capture_positions = []
    sel = logits[:, LETTER_IDS[:n_opt]].float()
    return torch.softmax(sel, dim=1).cpu().numpy()  # [B, n_opt]


## 4. Pass 1 — baseline + donor di posisi identitas (BATCHED)

Kenapa loop lama lambat: tiap prompt dijalankan SATU-SATU (batch=1) -- GPU
memang dipakai, tapi kerjaannya kecil banget per panggilan dibanding overhead
Python/tokenizer/hook di sekitarnya, jadi GPU banyak nganggur nunggu giliran.

Percepatan: semua kelompok (`gk`) yang berbagi PERTANYAAN (`qk`) yang sama
punya prompt dengan PANJANG TOKEN IDENTIK (bedanya cuma 1 token huruf jawaban
identitas) -- jadi bisa digabung jadi SATU batch, satu forward pass, alih-alih
satu-satu. Ada pengecekan aman: kalau ternyata ada prompt yang panjangnya beda
(seharusnya tidak, tapi dicek biar tidak diam-diam salah), grup itu jatuh balik
ke jalur lama (satu-satu) -- tidak pernah silently salah.


In [ ]:
BASELINE_BATCH_SIZE = 16  # aman di VRAM T4 utk prompt pendek; turunkan kalau OOM

baseline_pred = {}
donors = {}      # (ty, gk, qk) -> {layer: {pos: vec}}
id_pos = {}      # (ty, qk) -> positions (sama utk semua sel se-tipe per soal)

for ty in TYPES_RUN:
    p = plan[ty]
    # kelompokkan (gk, qk) yang belum diproses, per qk -- ini kuncinya batching
    by_qk = {}
    for (gk, qk) in p["needed"]:
        if (ty, gk, qk) in donors:
            continue
        by_qk.setdefault(qk, []).append(gk)

    for qk, gks in tqdm(by_qk.items(), desc=f"baseline {ty}"):
        prompts = [build_prompt(ty, gk, qk) for gk in gks]
        if (ty, qk) not in id_pos:
            id_pos[(ty, qk)] = identity_positions(prompts[0])
        positions = id_pos[(ty, qk)]
        n_opt = len(qmeta[qk][2])

        # safety check: semua prompt di grup ini harus sama panjang token
        lens = [len(tokenizer.encode(pr, add_special_tokens=False)) for pr in prompts]

        if len(set(lens)) == 1:
            # jalur cepat: gabung semua gk jadi batch (dipecah biar tidak OOM)
            for start in range(0, len(gks), BASELINE_BATCH_SIZE):
                batch_gks = gks[start:start + BASELINE_BATCH_SIZE]
                batch_prompts = prompts[start:start + BASELINE_BATCH_SIZE]
                preds = forward_pred_batch(batch_prompts, n_opt, positions)
                for b, gk in enumerate(batch_gks):
                    baseline_pred[(gk, qk)] = preds[b]
                    donors[(ty, gk, qk)] = {
                        L: {pos: _donor_capture[L][pos][b].clone() for pos in positions}
                        for L in CAPTURE_LAYERS
                    }
        else:
            # fallback: panjang token beda (harusnya tidak terjadi) -> satu-satu, aman
            print(f"  [{ty} / {qk}] panjang token tidak seragam {set(lens)} -> fallback per-item")
            for gk, pr in zip(gks, prompts):
                pred = forward_pred(pr, n_opt, capture_positions=positions)
                baseline_pred[(gk, qk)] = pred
                donors[(ty, gk, qk)] = {
                    L: {pos: _donor_capture[L][pos][0].clone() for pos in positions}
                    for L in CAPTURE_LAYERS
                }

print(f"{len(baseline_pred)} baseline selesai.")


## 5. Pass 2 — kondisi patch di posisi identitas

In [ ]:
ALL_HEADS = list(range(NUM_HEADS))

def make_spec(ty, B, qk, layer_heads_alpha):
    """layer_heads_alpha: list of (layer, heads, alpha). Donor = milik B."""
    positions = id_pos[(ty, qk)]
    spec = {}
    for (L, heads, alpha) in layer_heads_alpha:
        spec.setdefault(L, [])
        for pos in positions:
            spec[L].append((pos, heads, alpha, donors[(ty, B, qk)][L][pos]))
    return spec

CONDITIONS = {
    "patch_L11_all32":    [(STAR_LAYER, ALL_HEADS, 1.0)],
    "patch_L11H16":       [(STAR_LAYER, [STAR_HEAD], 1.0)],
    "patch_L11L18_all":   [(STAR_LAYER, ALL_HEADS, 1.0), (SECOND_LAYER, ALL_HEADS, 1.0)],
    "patch_L11_all32_x5": [(STAR_LAYER, ALL_HEADS, 5.0)],
    "patch_randhead":     [(RAND_HEAD[0], [RAND_HEAD[1]], 1.0)],
}

def wd(pred, real, ordinal):
    return wasserstein_distance(ordinal, ordinal, u_weights=pred, v_weights=real)

rows = []
for ty in TYPES_RUN:
    p = plan[ty]
    for pi, (A, B) in enumerate(tqdm(p["pairs"], desc=f"patch {ty}")):
        for qk in p["pair_questions"][(A, B)]:
            question, options, ordinal = qmeta[qk]
            n_opt = len(ordinal)
            realA, realB = real_resp[(A, qk)], real_resp[(B, qk)]
            predA, predB = baseline_pred[(A, qk)], baseline_pred[(B, qk)]
            prompt_A = build_prompt(ty, A, qk)
            base = dict(attr_type=ty, pair=f"{A} -> {B}", qkey=qk,
                        wd_A_to_realA=wd(predA, realA, ordinal),
                        wd_A_to_realB=wd(predA, realB, ordinal),
                        wd_B_to_realB=wd(predB, realB, ordinal),
                        wd_predA_predB=wd(predA, predB, ordinal),
                        real_wd_AB=real_wd(A, B, qk))
            for cond, lha in CONDITIONS.items():
                pp = forward_pred(prompt_A, n_opt, patch_spec=make_spec(ty, B, qk, lha))
                rows.append(dict(base, condition=cond,
                                 wd_patch_to_realB=wd(pp, realB, ordinal),
                                 wd_patch_to_realA=wd(pp, realA, ordinal),
                                 wd_patch_to_predB=wd(pp, predB, ordinal)))
            if pi < N_SELF_PATCH:
                spec = make_spec(ty, A, qk, [(STAR_LAYER, ALL_HEADS, 1.0)])
                pp = forward_pred(prompt_A, n_opt, patch_spec=spec)
                rows.append(dict(base, condition="self_patch",
                                 wd_patch_to_realB=wd(pp, realB, ordinal),
                                 wd_patch_to_realA=wd(pp, realA, ordinal),
                                 wd_patch_to_predB=wd(pp, predB, ordinal)))

res = pd.DataFrame(rows)
res["shift_ke_realB"] = res["wd_A_to_realB"] - res["wd_patch_to_realB"]
res["shift_ke_predB"] = res["wd_predA_predB"] - res["wd_patch_to_predB"]  # gerak mekanis ke prediksi B
res.to_csv(os.path.join(OUT_DIR, "patching_v2_rows.csv"), index=False)
print(res.shape, "-> patching_v2_rows.csv")


## 6. Rekap + uji statistik

Dua skor: `shift_ke_realB` (kesetiaan — mendekat ke distribusi ASLI B) dan
`shift_ke_predB` (mekanis — mendekat ke PREDIKSI B; ini cek apakah patch-nya
"bekerja" sama sekali). Patch bisa saja bekerja mekanis tapi tidak setia —
itu justru pemisahan yang kita mau.


In [ ]:
summary_rows = []
print("=" * 110)
for ty in TYPES_RUN:
    r_ty = res[res["attr_type"] == ty]
    ceiling = (r_ty.drop_duplicates(["pair", "qkey"])["wd_A_to_realB"]
               - r_ty.drop_duplicates(["pair", "qkey"])["wd_B_to_realB"]).mean()
    print(f"\n{ty} — langit-langit full-swap (mean): {ceiling:+.4f}")
    rand = r_ty[r_ty["condition"] == "patch_randhead"].set_index(["pair", "qkey"])["shift_ke_realB"]
    for cond in list(CONDITIONS) + ["self_patch"]:
        s_all = r_ty[r_ty["condition"] == cond].set_index(["pair", "qkey"])
        if len(s_all) == 0:
            continue
        s, sm = s_all["shift_ke_realB"], s_all["shift_ke_predB"]
        p_vs_rand = np.nan
        if cond not in ("patch_randhead", "self_patch"):
            joined = pd.concat([s, rand], axis=1, keys=["c", "r"]).dropna()
            if len(joined) > 10 and not np.allclose(joined["c"], joined["r"]):
                p_vs_rand = float(wilcoxon(joined["c"], joined["r"]).pvalue)
        summary_rows.append(dict(attr_type=ty, condition=cond, n=len(s),
                                 ceiling_full_swap=float(ceiling),
                                 mean_shift_ke_realB=float(s.mean()),
                                 persen_positif=float((s > 0).mean()),
                                 mean_shift_ke_predB=float(sm.mean()),
                                 p_wilcoxon_vs_random=p_vs_rand))
        p_str = "-" if np.isnan(p_vs_rand) else f"{p_vs_rand:.4f}"
        print(f"  {cond:20s} n={len(s):4d} | shift->realB={s.mean():+.4f} (>0: {(s>0).mean():.0%}) | "
              f"shift->predB={sm.mean():+.4f} | p vs random: {p_str}")

summary = pd.DataFrame(summary_rows)
summary.to_csv(os.path.join(OUT_DIR, "patching_v2_summary.csv"), index=False)

print("\nKonteks baseline per tipe:")
for ty in TYPES_RUN:
    b = res[res["attr_type"] == ty].drop_duplicates(["pair", "qkey"])
    print(f"{ty:16s} WD(predA,predB)={b['wd_predA_predB'].mean():.4f} | "
          f"WD(predA,realB)={b['wd_A_to_realB'].mean():.4f} | "
          f"WD(predB,realB)={b['wd_B_to_realB'].mean():.4f} | "
          f"WD asli antar kelompok={b['real_wd_AB'].mean():.4f}")


## Cara baca hasil

Urutan pengecekan:

1. **Patch bekerja mekanis?** `shift_ke_predB` untuk `patch_L11_all32` harus > 0
   dan jauh di atas `patch_randhead`. Kalau nol juga -> identitas TIDAK dibawa
   lewat attention output L11 di posisi itu (informasi lewat jalur lain/MLP).
2. **Bekerja DAN setia?** `shift_ke_realB` > 0, p < 0.05 vs random -> L11 punya
   peran kausal yang benar secara demografis. Bandingkan magnitudo dengan
   `ceiling_full_swap` (berapa % langit-langit yang tercapai).
3. `patch_L11H16` vs `patch_L11_all32`: 1 head cukup atau butuh koor 32 head?
4. `x5` vs `x1`: kalau cuma x5 yang gerak -> efeknya butuh amplifikasi
   (senada llm-opinions yang pakai clamp 5-15x) — output ada tapi "teredam".
5. `self_patch` wajib ~0; `randhead` wajib ~0.
6. Baris konteks: kalau `WD(predA,predB)` tetap kecil bahkan dengan prompt QA +
   soal ber-WD-asli terbesar -> insensitivitas output (finding 07 D1-D2)
   terkonfirmasi juga di format prompt mereka. Itu temuan penting tersendiri.

**Download:** `patching_v2_rows.csv` + `patching_v2_summary.csv` ->
`notebooks/output/12_tahap2_patching_v2_kaggle/`. Hasil -> finding 08.
